# 05 - Saving the model so it can leave this notebook
BCS 3101 Capstone: predicting telecom customer churn


This notebook does the smallest possible version of that:
1. Train the final model (the same one from notebook 04) on the full training set.
2. Save it to one file with `joblib`.
3. **Pretend to be a completely different program**: load that file back and use it, without
   retraining anything.
4. Run one prediction on a made-up customer, to prove the saved file actually works on its own.

## Step 1: Train the final model, one more time

In [1]:
import sys, warnings
sys.path.append('..')
warnings.filterwarnings('ignore')

import pandas as pd
import joblib

from src.setup import prepare_everything
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test, preprocessing, cv = prepare_everything()

final_model = Pipeline([
    ('preprocessing', preprocessing),
    ('model', LogisticRegression(C=100, max_iter=1000, random_state=42)),
])
final_model.fit(X_train, y_train)

print('Model trained.')

Model trained.


## Step 2: Save it to one file
`joblib` is the standard way to save a trained scikit-learn model. It saves EVERYTHING the pipeline
learned - the median it fills missing values with, the scaling numbers, the one-hot encoder's list
of categories, and the logistic regression's weights - all in one file, so nothing needs to be
re-learned later.

In [2]:
joblib.dump(final_model, '../models/final_pipeline.joblib')
print('Saved to ../models/final_pipeline.joblib')

Saved to ../models/final_pipeline.joblib


## Step 3: Pretend to be a different program
Below, we load the saved file back using `joblib.load`. To prove this really works on its own, we do
NOT reuse `final_model` from above - we load a fresh copy into a new variable name, as if this code
were running in a completely separate script that has never seen the training process.

In [3]:
loaded_model = joblib.load('../models/final_pipeline.joblib')
print('Loaded successfully. Type of object:', type(loaded_model).__name__)

Loaded successfully. Type of object: Pipeline


## Step 4: Run one prediction on a made-up customer
This is a single row of data shaped exactly like the columns the model expects - a new customer who
was never in the training or test set. If the loaded model can predict on this without errors, we've
proven it works independently of this notebook.

In [4]:
new_customer = pd.DataFrame([{
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 2,                        # a brand-new customer
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 95.0,
    'TotalCharges': 190.0,
}])

prediction = loaded_model.predict(new_customer)[0]
probability = loaded_model.predict_proba(new_customer)[0, 1]

print('Prediction:', 'CHURN' if prediction == 1 else 'NO CHURN')
print(f'Estimated churn probability: {probability:.1%}')

Prediction: CHURN
Estimated churn probability: 70.4%
